# 02. 분석 테이블 구축과 검증

## 분석 개요

- **역할**: 세션, 세션·상품, 대표 첫 구매 분석용 세션·상품 분석 테이블의 생성 정의와 검증 계약을 관리한다.
- **분석 단위**: `mart_session`은 `user_session`, 나머지 두 분석 테이블은 `user_session × product_id` 1행이다.
- **기본 실행**: 기존 분석 테이블을 재사용하고 정확한 행 수·필수 스키마·인덱스·기본 논리만 기본 점검한다.
- **전체 재검증**: 원본 재집계는 명시적으로 활성화했을 때만 수행하며 공통 `QueryCache`를 사용한다.
- **제약**: 기본값에서는 분석 테이블 생성문과 장시간 원본 대조를 실행하지 않는다.

In [1]:
import hashlib
import json
import os
import re
import sys
import time
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import quote_plus

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, event, text

PROJECT_ROOT = next(
    (base for base in (Path.cwd(), *Path.cwd().parents)
     if (base / 'cache_context.json').is_file() and (base / 'sql').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(f'프로젝트 루트를 찾을 수 없다 (cwd={Path.cwd()})')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / '.env')
engine = create_engine(
    f"mysql+pymysql://{os.getenv('DB_USER')}:{quote_plus(os.getenv('DB_PASSWORD'))}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}?charset=utf8mb4"
)

# 실행 모드는 이 셀 한곳에서만 설정한다.
REBUILD_SESSION_MARTS = False
REBUILD_JOURNEY_MART = False
RUN_FULL_VALIDATION = False
RUN_SMOKE_CHECK = True
MART_REBUILD_CONFIRMATION = ''

In [2]:
from query_cache import QueryCache, load_queries

SQL_FILE = PROJECT_ROOT / 'sql' / '02_preprocessing_mart.sql'
Q = load_queries(SQL_FILE)
EXPECTED_QUERIES = [
    'create_mart_session',
    'raw_session_reconciliation',
    'mart_session_reconciliation',
    'create_mart_session_product',
    'raw_session_product_reconciliation',
    'mart_session_product_reconciliation',
    'create_mart_user_product_session',
    'raw_user_product_session_reconciliation',
    'raw_user_product_before_purchase_reconciliation',
    'mart_user_product_session_reconciliation',
    'mart_inventory_smoke',
    'mart_schema_contract',
]
if list(Q) != EXPECTED_QUERIES:
    raise ValueError(f'named query 구성이 다르다: {list(Q)}')

QUERY_CACHE = QueryCache(
    engine=engine,
    sql_file=SQL_FILE,
    upstream_sql_files=(),
)
executed_sql_types = []

def _classify_sql_type(statement):
    cleaned = re.sub(r'^\s*(?:/\*.*?\*/\s*)*', '', statement, flags=re.DOTALL)
    match = re.match(r'^(\w+)', cleaned)
    statement_type = match.group(1).upper() if match else 'UNKNOWN'
    if statement_type == 'WITH':
        write_match = re.search(
            r'\b(INSERT|UPDATE|DELETE|REPLACE|CREATE|DROP|ALTER|TRUNCATE)\b',
            cleaned, flags=re.IGNORECASE,
        )
        return write_match.group(1).upper() if write_match else 'SELECT'
    return statement_type

@event.listens_for(engine, 'before_cursor_execute')
def _record_sql_type(conn, cursor, statement, parameters, context, executemany):
    executed_sql_types.append(_classify_sql_type(statement))

def split_statements(name):
    body = re.sub(r'--[^\n]*', '', Q[name])
    return [statement.strip() for statement in body.split(';') if statement.strip()]

REBUILD_REQUESTED = REBUILD_SESSION_MARTS is True or REBUILD_JOURNEY_MART is True
REBUILD_ALLOWED = (
    REBUILD_REQUESTED
    and MART_REBUILD_CONFIRMATION == 'DROP_AND_REBUILD_MARTS'
    and RUN_FULL_VALIDATION is True
)
if REBUILD_REQUESTED and not REBUILD_ALLOWED:
    raise RuntimeError(
        '마트 재생성에는 재생성 스위치, 정확한 확인 문자열, '
        'RUN_FULL_VALIDATION=True가 모두 필요하다.'
    )

def execute_write(name):
    if not REBUILD_ALLOWED:
        raise PermissionError('안전 조건 미충족: 생성 SQL을 실행할 수 없다.')
    with engine.begin() as conn:
        for statement in split_statements(name):
            conn.execute(text(statement))

VALIDATION_QUERY_NAMES = [
    'raw_session_reconciliation', 'mart_session_reconciliation',
    'raw_session_product_reconciliation', 'mart_session_product_reconciliation',
    'raw_user_product_session_reconciliation',
    'raw_user_product_before_purchase_reconciliation',
    'mart_user_product_session_reconciliation',
]
validation_timings = {}

def run_validation(name):
    if not RUN_FULL_VALIDATION:
        raise PermissionError('RUN_FULL_VALIDATION=False: raw full validation을 실행할 수 없다.')
    if name in validation_timings:
        raise RuntimeError(f'같은 검증 쿼리를 중복 실행할 수 없다: {name}')
    started_at = time.perf_counter()
    frame = QUERY_CACHE.run(name, refresh=True)
    validation_timings[name] = round(time.perf_counter() - started_at, 3)
    return frame

---
## 1. 안전 스위치와 DB 연결

재생성 스위치 중 하나 이상, 확인 문자열 `DROP_AND_REBUILD_MARTS`, `RUN_FULL_VALIDATION=True`가 모두 충족돼야 생성 SQL을 실행한다. 기본 모드는 이 조건을 충족하지 않으며 기존 분석 테이블만 읽는다.

In [3]:
DDL_EXPECTED_HASHES = {
    'create_mart_session': 'e9d17d6dd0eefd69bd9fa7c23799ad0f7496675deebaaa5392193a74ff4e2001',
    'create_mart_session_product': '3644f342a8cff9736a5f434b234be258c0811c4c69f769a5d39a60c03e578832',
    'create_mart_user_product_session': '557a0dd6bb2af0d8a2ee880f09834847e917149b2c6f48341d8dc2adc28d6308',
}
ddl_audit = []
for name, expected_hash in DDL_EXPECTED_HASHES.items():
    statements = split_statements(name)
    statement_types = [re.match(r'^(\w+)', statement).group(1).upper() for statement in statements]
    current_hash = hashlib.sha256(Q[name].encode()).hexdigest()
    ddl_audit.append({
        'query': name,
        '문장수': len(statements),
        '문장유형': ', '.join(sorted(set(statement_types))),
        'DDL_hash': current_hash[:12],
        '기존정의_일치': current_hash == expected_hash,
    })
ddl_audit = pd.DataFrame(ddl_audit)
if not bool(ddl_audit['기존정의_일치'].all()):
    raise AssertionError('생성 DDL이 승인된 정의와 달라졌다.')

mode_status = pd.DataFrame([{
    'REBUILD_SESSION_MARTS': REBUILD_SESSION_MARTS,
    'REBUILD_JOURNEY_MART': REBUILD_JOURNEY_MART,
    'RUN_FULL_VALIDATION': RUN_FULL_VALIDATION,
    'RUN_SMOKE_CHECK': RUN_SMOKE_CHECK,
    '재생성_허용': REBUILD_ALLOWED,
}])
display(mode_status)
ddl_audit

,REBUILD_SESSION_MARTS,REBUILD_JOURNEY_MART,RUN_FULL_VALIDATION,RUN_SMOKE_CHECK,재생성_허용
0,False,False,False,True,False


,query,문장수,문장유형,DDL_hash,기존정의_일치
0,create_mart_session,18,"ALTER, CREATE, DROP, SET",e9d17d6dd0ee,True
1,create_mart_session_product,16,"ALTER, CREATE, DROP",3644f342a8cf,True
2,create_mart_user_product_session,14,"ALTER, CREATE, DROP",557a0dd6bb2a,True


---
## 2. 세션·세션상품 분석 테이블 생성 분기

`REBUILD_SESSION_MARTS=True`이고 이중 확인 및 전체 재검증 조건까지 충족된 경우에만 두 생성 쿼리를 순서대로 실행한다. 기본 실행에서는 두 쿼리를 파싱만 하고 건너뛴다.

In [4]:
session_build_log = []
if REBUILD_SESSION_MARTS:
    for query_name, table_name in [
        ('create_mart_session', 'mart_session'),
        ('create_mart_session_product', 'mart_session_product'),
    ]:
        started_at = time.perf_counter()
        execute_write(query_name)
        session_build_log.append({
            'table': table_name,
            '상태': '재생성',
            '소요초': round(time.perf_counter() - started_at, 1),
        })
else:
    session_build_log = [
        {'table': 'mart_session', '상태': '기존 마트 재사용', '소요초': 0.0},
        {'table': 'mart_session_product', '상태': '기존 마트 재사용', '소요초': 0.0},
    ]
pd.DataFrame(session_build_log)

,table,상태,소요초
0,mart_session,기존 마트 재사용,0.0
1,mart_session_product,기존 마트 재사용,0.0


---
## 3. 세션·세션상품 통합 검증 분기

전체 재검증은 원본 유효 세션·이벤트 카운트·revenue·순서가 확인된 건·동시 시각까지 포함한 건과 분석 테이블 카운트·플래그·논리를 대조한다. 세션·상품도 같은 구조로 행 수·복합 단위·이벤트 카운트·순차 플래그를 검증한다. 기본 실행에서는 실행하지 않는다.

In [5]:
if RUN_FULL_VALIDATION:
    raw_session = run_validation('raw_session_reconciliation').iloc[0]
    mart_session = run_validation('mart_session_reconciliation').iloc[0]
    raw_msp = run_validation('raw_session_product_reconciliation').iloc[0]
    mart_msp = run_validation('mart_session_product_reconciliation').iloc[0]

    session_pairs = [
        ('행수', 'raw_유효세션수', '마트_행수'),
        ('view', 'raw_views', '마트_views'), ('cart', 'raw_carts', '마트_carts'),
        ('remove', 'raw_removes', '마트_removes'),
        ('purchase', 'raw_purchases', '마트_purchases'),
        ('view 단위', 'view_units', 'view_units'),
        ('strict stage2', 'strict_stage2', 'strict_stage2'),
        ('strict stage3', 'strict_stage3', 'strict_stage3'),
    ]
    msp_pairs = [
        ('행수', 'raw_행수', '마트_행수'),
        ('view', 'raw_views', '마트_views'), ('cart', 'raw_carts', '마트_carts'),
        ('purchase', 'raw_purchases', '마트_purchases'),
        ('view 단위', 'view_units', 'view_units'),
        ('strict stage2', 'strict_stage2', 'strict_stage2'),
        ('strict stage3', 'strict_stage3', 'strict_stage3'),
    ]
    session_checks = pd.DataFrame([
        {'범위': 'session', '지표': label, 'raw': float(raw_session[raw_key]), 'mart': float(mart_session[mart_key])}
        for label, raw_key, mart_key in session_pairs
    ] + [
        {'범위': 'session_product', '지표': label, 'raw': float(raw_msp[raw_key]), 'mart': float(mart_msp[mart_key])}
        for label, raw_key, mart_key in msp_pairs
    ])
    session_checks['일치'] = session_checks['raw'].round(4) == session_checks['mart'].round(4)
    logic_columns = [column for column in mart_session.index if 'invalid' in column or 'without' in column]
    msp_logic_columns = [column for column in mart_msp.index if 'invalid' in column or 'without' in column]
    session_validation_gate = (
        bool(session_checks['일치'].all())
        and round(float(raw_session['raw_revenue']), 2) == round(float(mart_session['마트_revenue']), 2)
        and int(mart_session['기본키_중복']) == 0
        and int(mart_msp['복합키_중복']) == 0
        and all(int(mart_session[column]) == 0 for column in logic_columns)
        and all(int(mart_msp[column]) == 0 for column in msp_logic_columns)
    )
    display(session_checks)
    print(f'세션·세션상품 full validation 통과: {session_validation_gate}')
    session_validation_status = pd.DataFrame([{
        '상태': '실행', '통과': session_validation_gate,
    }])
else:
    session_validation_gate = None
    session_validation_status = pd.DataFrame([{
        '상태': '생략', '사유': 'RUN_FULL_VALIDATION=False',
    }])
session_validation_status

,상태,사유
0,생략,RUN_FULL_VALIDATION=False


---
## 4. 대표 첫 구매용 분석 테이블 생성 분기

`mart_user_product_session`은 05의 대표 첫 구매 경로와 cart 이후 구매율에 필요한 이벤트 시각을 보존한다. `REBUILD_JOURNEY_MART=True`와 모든 안전 조건을 충족했을 때만 재생성한다.

In [6]:
if REBUILD_JOURNEY_MART:
    started_at = time.perf_counter()
    execute_write('create_mart_user_product_session')
    journey_build_status = pd.DataFrame([{
        'table': 'mart_user_product_session',
        '상태': '재생성',
        '소요초': round(time.perf_counter() - started_at, 1),
    }])
else:
    journey_build_status = pd.DataFrame([{
        'table': 'mart_user_product_session',
        '상태': '기존 마트 재사용',
        '소요초': 0.0,
    }])
journey_build_status

,table,상태,소요초
0,mart_user_product_session,기존 마트 재사용,0.0


---
## 5. 대표 첫 구매용 분석 테이블 통합 검증 분기

원본 이벤트 로그와 분석 테이블의 행 수·이벤트 카운트·revenue·최초/최종 시각을 대조하고, 최초 구매 전 마지막 행동 시각과 분석 테이블 내부 카운트·시각·플래그 논리를 검증한다. 기본 실행에서는 실행하지 않는다.

In [7]:
if RUN_FULL_VALIDATION:
    raw_journey = run_validation('raw_user_product_session_reconciliation').iloc[0]
    raw_before = run_validation('raw_user_product_before_purchase_reconciliation').iloc[0]
    mart_journey = run_validation('mart_user_product_session_reconciliation').iloc[0]
    journey_pairs = [
        ('행수', 'raw_행수', '마트_행수'),
        ('view', 'raw_views', '마트_views'), ('cart', 'raw_carts', '마트_carts'),
        ('remove', 'raw_removes', '마트_removes'),
        ('purchase', 'raw_purchases', '마트_purchases'),
    ]
    journey_checks = pd.DataFrame([
        {'지표': label, 'raw': float(raw_journey[raw_key]), 'mart': float(mart_journey[mart_key])}
        for label, raw_key, mart_key in journey_pairs
    ])
    journey_checks['일치'] = journey_checks['raw'].round(4) == journey_checks['mart'].round(4)
    raw_error_columns = [
        '마트누락_행수', 'user_id_불일치', '카운트_불일치', '시각_불일치', 'revenue_불일치',
    ]
    mart_logic_columns = [column for column in mart_journey.index if '오류' in column or '역전' in column]
    journey_validation_gate = (
        bool(journey_checks['일치'].all())
        and round(float(raw_journey['raw_revenue']), 2) == round(float(mart_journey['마트_revenue']), 2)
        and all(int(raw_journey[column]) == 0 for column in raw_error_columns)
        and int(raw_before['raw_구매전행동_행수']) == int(mart_journey['마트_구매행수'])
        and int(raw_before['구매전행동시각_불일치']) == 0
        and int(mart_journey['복합키_중복']) == 0
        and all(int(mart_journey[column]) == 0 for column in mart_logic_columns)
    )
    display(journey_checks)
    print(f'대표 첫 구매용 마트 full validation 통과: {journey_validation_gate}')
    journey_validation_status = pd.DataFrame([{
        '상태': '실행', '통과': journey_validation_gate,
    }])
else:
    journey_validation_gate = None
    journey_validation_status = pd.DataFrame([{
        '상태': '생략', '사유': 'RUN_FULL_VALIDATION=False',
    }])
journey_validation_status

,상태,사유
0,생략,RUN_FULL_VALIDATION=False


---
## 6. 기본 점검

캐시를 사용하지 않고 현재 DB에서 세 분석 테이블의 존재, 정확한 행 수, 필수 컬럼, PK·하류 인덱스, 기본 논리 위반을 직접 확인한다. 직전 실행 행 수는 비교 기준일 뿐 조회 결과를 대체하지 않는다.

In [8]:
EXPECTED_TABLES = {
    'mart_session', 'mart_session_product', 'mart_user_product_session',
}
smoke_timings = {}
if RUN_SMOKE_CHECK:
    table_rows = pd.read_sql(text('SHOW TABLES'), engine)
    existing_tables = set(table_rows.iloc[:, 0].astype(str))
    missing_tables = sorted(EXPECTED_TABLES - existing_tables)
    if missing_tables:
        raise RuntimeError(f'필수 마트가 없다: {missing_tables}')
    started_at = time.perf_counter()
    mart_inventory = pd.read_sql(text(Q['mart_inventory_smoke']), engine)
    smoke_timings['mart_inventory_smoke'] = round(time.perf_counter() - started_at, 3)
    started_at = time.perf_counter()
    schema_contract = pd.read_sql(text(Q['mart_schema_contract']), engine)
    smoke_timings['mart_schema_contract'] = round(time.perf_counter() - started_at, 3)
else:
    mart_inventory = pd.DataFrame()
    schema_contract = pd.DataFrame()
    missing_tables = []
pd.DataFrame([{
    'RUN_SMOKE_CHECK': RUN_SMOKE_CHECK,
    '누락_마트': ', '.join(missing_tables) or '없음',
    **{f'{name}_초': seconds for name, seconds in smoke_timings.items()},
}])

,RUN_SMOKE_CHECK,누락_마트,mart_inventory_smoke_초,mart_schema_contract_초
0,True,없음,26.485,0.01


In [9]:
ROW_REFERENCES = {
    'mart_session': 4_499_479,
    'mart_session_product': 12_102_048,
    'mart_user_product_session': 13_385_787,
}
REQUIRED_COLUMNS = {
    'mart_session': {
        'user_session', 'user_id', 'views', 'carts', 'purchases',
        'has_cart_after_view', 'has_purchase_after_view_cart',
    },
    'mart_session_product': {
        'user_session', 'product_id', 'views', 'carts', 'purchases',
        'has_cart_after_view', 'has_purchase_after_view_cart',
    },
    'mart_user_product_session': {
        'user_session', 'user_id', 'product_id', 'session_start', 'session_end',
        'carts', 'purchases', 'first_view_at', 'first_cart_at', 'last_cart_at',
        'first_purchase_at', 'last_purchase_at', 'last_cart_before_first_purchase_at',
    },
}
REQUIRED_INDEXES = {
    ('mart_session', 'PRIMARY'): ('user_session',),
    ('mart_session', 'idx_mart_user'): ('user_id',),
    ('mart_session_product', 'PRIMARY'): ('user_session', 'product_id'),
    ('mart_user_product_session', 'PRIMARY'): ('user_session', 'product_id'),
    ('mart_user_product_session', 'idx_mups_user_product_start'): (
        'user_id', 'product_id', 'session_start', 'user_session',
    ),
    ('mart_user_product_session', 'idx_mups_first_purchase'): ('first_purchase_at',),
}

if RUN_SMOKE_CHECK:
    mart_inventory['회귀_참고행수'] = mart_inventory['table_name'].map(ROW_REFERENCES)
    mart_inventory['행수_일치'] = mart_inventory['row_count'] == mart_inventory['회귀_참고행수']
    column_rows = schema_contract[schema_contract['contract_type'] == 'COLUMN']
    actual_columns = column_rows.groupby('mart_table')['contract_column'].apply(set).to_dict()
    column_checks = pd.DataFrame([
        {
            'table_name': table_name,
            '누락_필수컬럼': ', '.join(sorted(required - actual_columns.get(table_name, set()))) or '없음',
        }
        for table_name, required in REQUIRED_COLUMNS.items()
    ])
    index_rows = schema_contract[schema_contract['contract_type'] == 'INDEX'].copy()
    actual_indexes = (
        index_rows.sort_values(['mart_table', 'object_name', 'contract_position'])
        .groupby(['mart_table', 'object_name'])['contract_column'].apply(tuple).to_dict()
    )
    index_checks = pd.DataFrame([
        {
            'table_name': key[0], 'index_name': key[1],
            '기대_컬럼': ', '.join(expected),
            '실제_컬럼': ', '.join(actual_indexes.get(key, ())),
            '일치': actual_indexes.get(key) == expected,
        }
        for key, expected in REQUIRED_INDEXES.items()
    ])
    smoke_gate = (
        bool(mart_inventory['행수_일치'].all())
        and bool((mart_inventory['basic_logic_violations'] == 0).all())
        and bool((column_checks['누락_필수컬럼'] == '없음').all())
        and bool(index_checks['일치'].all())
    )
    if not smoke_gate:
        raise AssertionError('마트 smoke check에 실패했다.')
    display(mart_inventory)
    display(column_checks)
    display(index_checks)
    smoke_status = pd.DataFrame([{'상태': '통과', 'smoke_gate': smoke_gate}])
else:
    smoke_gate = None
    smoke_status = pd.DataFrame([{'상태': '생략', '사유': 'RUN_SMOKE_CHECK=False'}])
smoke_status

,table_name,row_count,basic_logic_violations,회귀_참고행수,행수_일치
0,mart_session,4499479,0.0,4499479,True
1,mart_session_product,12102048,0.0,12102048,True
2,mart_user_product_session,13385787,0.0,13385787,True


,table_name,누락_필수컬럼
0,mart_session,없음
1,mart_session_product,없음
2,mart_user_product_session,없음


,table_name,index_name,기대_컬럼,실제_컬럼,일치
0,mart_session,PRIMARY,user_session,user_session,True
1,mart_session,idx_mart_user,user_id,user_id,True
2,mart_session_product,PRIMARY,"user_session, product_id","user_session, product_id",True
3,mart_user_product_session,PRIMARY,"user_session, product_id","user_session, product_id",True
4,mart_user_product_session,idx_mups_user_product_start,"user_id, product_id, session_start, user_session","user_id, product_id, session_start, user_session",True
5,mart_user_product_session,idx_mups_first_purchase,first_purchase_at,first_purchase_at,True


,상태,smoke_gate
0,통과,True


---
## 7. 하류 실행 가능 상태와 03 인계

기본 점검을 통과하면 기존 세 분석 테이블이 03-05의 입력 계약을 충족한 것으로 본다. 이는 원본 전체 대조를 대신하지 않으며, 02-B에서 별도로 실행한다.

In [10]:
VALIDATION_MANIFEST = (
    PROJECT_ROOT / 'cache' / SQL_FILE.stem / 'full_validation_manifest.json'
)
if RUN_FULL_VALIDATION:
    full_validation_gate = (
        session_validation_gate is True
        and journey_validation_gate is True
        and smoke_gate is True
    )
    if not full_validation_gate:
        raise AssertionError('02 full validation에 실패했다. cache manifest를 확정하지 않는다.')
    mart_row_counts = {
        row['table_name']: int(row['row_count'])
        for row in mart_inventory[['table_name', 'row_count']].to_dict('records')
    }
    creation_query_hashes = {
        name: hashlib.sha256(Q[name].encode()).hexdigest()
        for name in DDL_EXPECTED_HASHES
    }
    cache_records = []
    for name in VALIDATION_QUERY_NAMES:
        fingerprint = QUERY_CACHE.fingerprint(name)
        metadata_path = (
            PROJECT_ROOT / 'cache' / SQL_FILE.stem / name / f'{fingerprint}.meta.json'
        )
        metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
        metadata.update({
            'raw_dataset_fingerprint': QUERY_CACHE.context['dataset_version'],
            'creation_query_sha256': creation_query_hashes,
            'mart_row_counts': mart_row_counts,
            'validation_elapsed_seconds': validation_timings[name],
        })
        metadata_temp = metadata_path.with_name(metadata_path.name + '.tmp')
        metadata_temp.write_text(
            json.dumps(metadata, ensure_ascii=False, indent=2, sort_keys=True),
            encoding='utf-8',
        )
        metadata_temp.replace(metadata_path)
        cache_records.append({
            'query': name, 'fingerprint': fingerprint,
            'elapsed_seconds': validation_timings[name],
            'created_at_utc': metadata['created_at_utc'],
            'row_count': metadata['row_count'],
            'content_hash': metadata['content_hash_sort_independent'],
        })
    manifest = {
        'completed_at_utc': datetime.now(timezone.utc).isoformat(),
        'all_passed': full_validation_gate,
        'dataset_fingerprint': QUERY_CACHE.context['dataset_version'],
        'sql_file_sha256': QUERY_CACHE.sql_file_sha256,
        'creation_query_sha256': creation_query_hashes,
        'mart_row_counts': mart_row_counts,
        'query_caches': cache_records,
        'total_query_seconds': round(sum(validation_timings.values()), 3),
    }
    VALIDATION_MANIFEST.parent.mkdir(parents=True, exist_ok=True)
    manifest_temp = VALIDATION_MANIFEST.with_name(VALIDATION_MANIFEST.name + '.tmp')
    manifest_temp.write_text(
        json.dumps(manifest, ensure_ascii=False, indent=2, sort_keys=True),
        encoding='utf-8',
    )
    manifest_temp.replace(VALIDATION_MANIFEST)
else:
    manifest = (
        json.loads(VALIDATION_MANIFEST.read_text(encoding='utf-8'))
        if VALIDATION_MANIFEST.exists() else None
    )

latest_validation_status = pd.DataFrame([{
    '최근_full_validation': manifest['completed_at_utc'] if manifest else '없음',
    '전체_통과': manifest['all_passed'] if manifest else False,
    '검증_cache수': len(manifest['query_caches']) if manifest else 0,
    '쿼리_총초': manifest['total_query_seconds'] if manifest else None,
}])

if not REBUILD_ALLOWED:
    unexpected_sql_types = sorted(set(executed_sql_types) - {'SELECT', 'SHOW'})
    if unexpected_sql_types:
        raise AssertionError(f'기본 모드에서 쓰기 가능 SQL이 실행됐다: {unexpected_sql_types}')

execution_audit = pd.DataFrame([{
    'named_query수': len(Q),
    'smoke_check_통과': smoke_gate,
    'full_validation_실행': RUN_FULL_VALIDATION,
    '재생성_실행': REBUILD_ALLOWED,
    '실행_SQL_유형': ', '.join(sorted(set(executed_sql_types))),
    '기본모드_쓰기_0건': (
        not REBUILD_ALLOWED
        and set(executed_sql_types).issubset({'SELECT', 'SHOW'})
    ),
}])
display(latest_validation_status)
execution_audit

,최근_full_validation,전체_통과,검증_cache수,쿼리_총초
0,2026-09-23T08:57:30.342275+00:00,True,7,2543.055


,named_query수,smoke_check_통과,full_validation_실행,재생성_실행,실행_SQL_유형,기본모드_쓰기_0건
0,12,True,False,False,"SELECT, SHOW",True


---
## 8. 캐시 영향과 다음 단계

- 02 SQL 변경으로 이 파일을 상위 출처 정보로 사용하는 04·05 캐시 지문은 무효화된다.
- 이번 02-A에서는 02 전체 재검증 캐시와 04·05 캐시를 생성하거나 갱신하지 않는다.
- 다음 02-B에서 분석 테이블을 재생성하지 않고 `RUN_FULL_VALIDATION=True`로 원본 대조만 실행한다.
- 검증을 통과한 뒤 03을 현재 하류 스토리에 맞게 축소한다.

> ### 기본 실행 결론
>
> - 세 분석 테이블을 재생성하지 않고 현재 DB의 행 수·필수 컬럼·PK·하류 인덱스·기본 논리를 확인한다.
> - 테이블 생성문은 기존 정의를 유지하며 명시적인 세 조건 없이는 실행되지 않는다.
> - 원본 전체 대조 결과는 현재 기본 출력에 섞지 않고 02-B에서 별도로 확정한다.